In [6]:
import os, re, glob, json, time
import numpy as np
import h5py
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
RAW_DIR = r"E:/rf_datasets_IQ_raw"   # 原始 .mat（含 rfDataset/dmrs）
EQ_DIR  = r"E:/rf_datasets_IQ_eq"    # 等化后 .mat（含 rfDataset/dmrs_eq）
RESULTS_ROOT = r"./results_lte_module2"

# -------------------------
# Source domain definition
# -------------------------
SOURCE_RX = [1,2]
SOURCE_SPEED = 10  # km/h, 固定，不做未知速度

# Known/Unknown TX split (可改)
KNOWN_TX = [1,2,3,4,5,6]  # 用于训练/原型
UNKNOWN_TX = [7,8,9]      # 仅用于 open-set 测试

# -------------------------
# Data subsampling (避免太大)
# -------------------------
MAX_SAMPLES_PER_FILE_TRAIN = 0   # 0=不截断；>0 每个文件最多用多少条训练样本
MAX_SAMPLES_PER_FILE_EVAL  = 0   # 0=不截断；>0 每个文件评估最多用多少条

# -------------------------
# Training hyperparams
# -------------------------
SEED = 42
BATCH_SIZE = 128
NUM_EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8

# -------------------------
# Model hyperparams
# -------------------------
IN_PLANES = 64
DROPOUT = 0.3
EMB_DIM = 512  # ResNet18最后一层前的embedding维度

# -------------------------
# Domain descriptor params
# -------------------------
NFFT_HALF = 128
OCC_START, OCC_END = 30, 101  # 你已经验证出的 occupied bins range（fftshift域）
OCC_LEN = OCC_END - OCC_START + 1  # 72
D_DIM_HALF = 16  # 每个128段提取16维domain特征
D_DIM = 2 * D_DIM_HALF

# -------------------------
# Scenario evaluation
# -------------------------
EVAL_RX_LIST_DRIFT = [3,4]  # 跨RX
EVAL_SPEED = 10              # 固定速度

# -------------------------
# Utils
# -------------------------
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(RESULTS_ROOT, f"run_{ts}")
os.makedirs(RUN_DIR, exist_ok=True)

CONFIG = {
    "RAW_DIR": RAW_DIR, "EQ_DIR": EQ_DIR, "RUN_DIR": RUN_DIR,
    "SOURCE_RX": SOURCE_RX, "SOURCE_SPEED": SOURCE_SPEED,
    "KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX,
    "BATCH_SIZE": BATCH_SIZE, "NUM_EPOCHS": NUM_EPOCHS, "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY, "PATIENCE": PATIENCE,
    "IN_PLANES": IN_PLANES, "DROPOUT": DROPOUT,
    "OCC_RANGE": [OCC_START, OCC_END], "D_DIM_HALF": D_DIM_HALF,
    "MAX_SAMPLES_PER_FILE_TRAIN": MAX_SAMPLES_PER_FILE_TRAIN,
    "MAX_SAMPLES_PER_FILE_EVAL": MAX_SAMPLES_PER_FILE_EVAL
}
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

Device: cuda


In [7]:
# ===== Robust filename parsing & MAT readers =====
# EQ directory may contain files WITHOUT .mat extension (Windows still shows "MATLAB Data").
# We therefore (1) parse by "stem" and (2) pair RAW/EQ by stem intersection.

fname_re = re.compile(
    r"^rfDataset_TX(?P<tx>\d+)_RX(?P<rx>\d+)_loc(?P<loc>\d+?)_(?P<sp>\d+)kmh_(?P<ts>\d{8}_\d{6})(?:_eq)?(?:\.mat)?$",
    re.IGNORECASE
)

def normalize_stem(name: str) -> str:
    """Remove optional extension and optional '_eq' suffix for pairing."""
    bn = name
    if bn.lower().endswith('.mat'):
        bn = bn[:-4]
    if bn.lower().endswith('_eq'):
        bn = bn[:-3]
    return bn

def parse_meta_from_stem(stem: str):
    m = fname_re.match(stem)
    if not m:
        raise ValueError(f"Stem not match pattern: {stem}")
    tx = int(m.group('tx'))
    rx = int(m.group('rx'))
    sp = int(m.group('sp'))
    return tx, rx, sp

def parse_meta_from_filename(fp: str):
    stem = normalize_stem(os.path.basename(fp))
    return parse_meta_from_stem(stem)


def read_compound_complex(arr):
    if isinstance(arr, np.ndarray) and arr.dtype.fields and ("real" in arr.dtype.fields) and ("imag" in arr.dtype.fields):
        return arr["real"] + 1j * arr["imag"]
    if np.iscomplexobj(arr):
        return arr
    return np.array(arr)


def load_dmrs_raw_Nx256(mat_path):
    with h5py.File(mat_path, "r") as f:
        rf = f["rfDataset"]
        dmrs_struct = rf["dmrs"][:]  # compound real/imag
        dmrs = read_compound_complex(dmrs_struct).astype(np.complex64)
    if dmrs.ndim != 2:
        raise ValueError(dmrs.shape)
    if dmrs.shape[1] == 256:
        return dmrs
    if dmrs.shape[0] == 256:
        return dmrs.T
    raise ValueError(f"Unexpected dmrs shape: {dmrs.shape}")


def load_dmrs_eq_Nx256(mat_path):
    # mat_path may have no extension; h5py can still open it.
    with h5py.File(mat_path, "r") as f:
        rf = f["rfDataset"]
        if "dmrs_eq" not in rf:
            raise KeyError(f"{os.path.basename(mat_path)}: rfDataset/dmrs_eq not found")
        dmrs_struct = rf["dmrs_eq"][:]
        dmrs = read_compound_complex(dmrs_struct).astype(np.complex64)
    if dmrs.ndim != 2:
        raise ValueError(dmrs.shape)
    if dmrs.shape[1] == 256:
        return dmrs
    if dmrs.shape[0] == 256:
        return dmrs.T
    raise ValueError(f"Unexpected dmrs_eq shape: {dmrs.shape}")

In [8]:
def fftshift_fft_1d(x):
    return np.fft.fftshift(np.fft.fft(x, n=NFFT_HALF))

def domain_feat_half(x128):
    """
    x128: (128,) complex
    return: (D_DIM_HALF,) float32
    """
    # 单位功率归一（稳定）；保留形状
    x = x128 / (np.sqrt(np.mean(np.abs(x128)**2)) + 1e-12)
    X = fftshift_fft_1d(x)
    mag = np.abs(X[OCC_START:OCC_END+1]) + 1e-12  # (72,)
    logm = np.log(mag)
    # 用 rfft 的低频实部当“DCT-like”系数
    D = np.fft.rfft(logm, n=2*len(logm))
    feat = np.real(D[:D_DIM_HALF]).astype(np.float32)
    return feat

def domain_feat_256(x256):
    """
    x256: (256,) complex raw dmrs
    return: (D_DIM,) float32
    """
    a = x256[:128]
    b = x256[128:256]
    fa = domain_feat_half(a)
    fb = domain_feat_half(b)
    return np.concatenate([fa, fb], axis=0).astype(np.float32)

In [9]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        # x: (B, 256, 2) -> (B,2,256)
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)  # (B,512)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [10]:
def complex256_to_float2(x256):
    # x256: (N,256) complex -> (N,256,2) float32
    return np.stack([x256.real, x256.imag], axis=-1).astype(np.float32)

# ===== Collect and pair RAW/EQ files by stem (EQ may have no .mat extension) =====
# RAW: only *.mat
raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.mat")))

# EQ: list all regular files (exclude meta json)
eq_paths = [p for p in glob.glob(os.path.join(EQ_DIR, "*")) if os.path.isfile(p)]
eq_paths = [p for p in eq_paths if not os.path.basename(p).lower().startswith("equalization_meta")]

raw_map = {}
for p in raw_paths:
    stem = normalize_stem(os.path.basename(p))
    try:
        _ = parse_meta_from_stem(stem)
    except Exception:
        continue
    raw_map[stem] = p

eq_map = {}
for p in eq_paths:
    stem = normalize_stem(os.path.basename(p))
    try:
        _ = parse_meta_from_stem(stem)
    except Exception:
        continue
    eq_map[stem] = p

common = sorted(set(raw_map.keys()) & set(eq_map.keys()))
print(f"[INFO] RAW files: {len(raw_paths)}  parsed: {len(raw_map)}")
print(f"[INFO]  EQ files: {len(eq_paths)}   parsed: {len(eq_map)}")
print(f"[INFO] Paired stems: {len(common)}")

file_records = []
for stem in common:
    tx, rx, sp = parse_meta_from_stem(stem)
    file_records.append({"stem": stem, "raw": raw_map[stem], "eq": eq_map[stem], "tx": tx, "rx": rx, "sp": sp})

# Quick sanity: list observed RX/Speeds
rxs = sorted({r["rx"] for r in file_records})
sps = sorted({r["sp"] for r in file_records})
txs = sorted({r["tx"] for r in file_records})
print("[INFO] TXs:", txs)
print("[INFO] RXs:", rxs)
print("[INFO] Speeds:", sps)

# ===== Source training files: source RX+speed, known TX =====
SOURCE_RXS = SOURCE_RX if isinstance(SOURCE_RX, (list, tuple, set)) else [SOURCE_RX]
train_files = [r for r in file_records if (r["rx"] in SOURCE_RXS and r["sp"]==SOURCE_SPEED and r["tx"] in KNOWN_TX)]
print("Train files:", len(train_files), "=>", [os.path.basename(r["raw"]) for r in train_files[:3]], "...")

if len(train_files) == 0:
    combos = sorted({(r["tx"], r["rx"], r["sp"]) for r in file_records})
    raise RuntimeError(
        "No train files found. Check SOURCE_RX/SOURCE_SPEED/KNOWN_TX. "
        f"Observed RXs={rxs}, Speeds={sps}, example combos={combos[:8]}"
    )

# ===== Load source train arrays =====
X_eq_list, y_list, d_list, file_id_list, row_id_list = [], [], [], [], []
for fi, rec in enumerate(train_files):
    dmrs_eq = load_dmrs_eq_Nx256(rec["eq"])
    dmrs_raw = load_dmrs_raw_Nx256(rec["raw"])
    assert dmrs_eq.shape == dmrs_raw.shape

    N = dmrs_eq.shape[0]
    if MAX_SAMPLES_PER_FILE_TRAIN and MAX_SAMPLES_PER_FILE_TRAIN > 0:
        N = min(N, MAX_SAMPLES_PER_FILE_TRAIN)
        dmrs_eq = dmrs_eq[:N]
        dmrs_raw = dmrs_raw[:N]

    X_eq_list.append(complex256_to_float2(dmrs_eq))
    y_list.append(np.full((N,), KNOWN_TX.index(rec["tx"]), dtype=np.int64))  # map to 0..K-1

    # domain feats (raw)
    d_this = np.stack([domain_feat_256(dmrs_raw[i]) for i in range(N)], axis=0)  # (N,D_DIM)
    d_list.append(d_this)

    file_id_list.append(np.full((N,), fi, dtype=np.int64))
    row_id_list.append(np.arange(N, dtype=np.int64))

X_eq = np.concatenate(X_eq_list, axis=0)
y = np.concatenate(y_list, axis=0)
D = np.concatenate(d_list, axis=0)
file_id = np.concatenate(file_id_list, axis=0)
row_id = np.concatenate(row_id_list, axis=0)

num_classes = len(KNOWN_TX)
print("Source train set:", X_eq.shape, y.shape, D.shape, "num_classes=", num_classes)

[INFO] RAW files: 72  parsed: 72
[INFO]  EQ files: 72   parsed: 72
[INFO] Paired stems: 72
[INFO] TXs: [1, 2, 3, 4, 5, 6, 7, 8, 9]
[INFO] RXs: [1, 2, 3, 4]
[INFO] Speeds: [10, 20]
Train files: 12 => ['rfDataset_TX001_RX001_loc01_10kmh_20260125_013420.mat', 'rfDataset_TX001_RX002_loc01_10kmh_20260125_013438.mat', 'rfDataset_TX002_RX001_loc01_10kmh_20260125_013535.mat'] ...
Source train set: (35989, 256, 2) (35989,) (35989, 32) num_classes= 6


In [11]:
BLOCK_SIZE = 256  # 可改；越大越保守，越小越接近随机

# 为每个样本分配 block id（按 file 内 row_id 分块）
block_id = (row_id // BLOCK_SIZE).astype(np.int64)
# 全局block key: (file_id, block_id)
block_key = file_id.astype(np.int64) * 10_000 + block_id  # 10k足够大防冲突

# 对每个 block 聚合出一个 label（该block的设备标签）
# 同一block内 label 应该一致（因为同一文件即同一tx）
unique_blocks, inv = np.unique(block_key, return_inverse=True)
block_label = np.zeros_like(unique_blocks, dtype=np.int64)
for bi, b in enumerate(unique_blocks):
    idx = np.where(block_key == b)[0]
    block_label[bi] = y[idx[0]]

print("Total blocks:", len(unique_blocks))

Total blocks: 144


In [12]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, total_correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()

        total_loss += float(loss.item()) * yb.size(0)
        pred = logits.argmax(dim=1)
        total_correct += int((pred == yb).sum().item())
        total += int(yb.size(0))

    return total_loss / max(1,total), total_correct / max(1,total)

def plot_curves(history, out_png):
    epochs = np.arange(1, len(history["train_loss"])+1)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, history["train_loss"], label="train")
    plt.plot(epochs, history["val_loss"], label="val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(True); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(epochs, history["train_acc"], label="train")
    plt.plot(epochs, history["val_acc"], label="val")
    plt.xlabel("epoch"); plt.ylabel("acc"); plt.grid(True); plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_confmat(cm, out_png, title="Confusion"):
    plt.figure(figsize=(6,5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xlabel("Pred"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

In [13]:
def compute_embeddings(model, X_np, batch_size=512):
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    Z = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            _, emb = model(xb, return_embed=True)
            Z.append(emb.cpu().numpy().astype(np.float32))
    Z = np.concatenate(Z, axis=0)  # (N,512)
    return Z

def l2_normalize(v, axis=1, eps=1e-12):
    n = np.linalg.norm(v, axis=axis, keepdims=True) + eps
    return v / n

def compute_prototypes(Z, y, num_classes):
    ZN = l2_normalize(Z, axis=1)
    P = np.zeros((num_classes, Z.shape[1]), dtype=np.float32)
    for k in range(num_classes):
        idx = np.where(y == k)[0]
        P[k] = np.mean(ZN[idx], axis=0)
    P = l2_normalize(P, axis=1)
    return P

def compute_domain_stats(D, y, num_classes, reg=1e-3):
    """
    For each class k: mu_k, Sigma_k (regularized)
    """
    mu = np.zeros((num_classes, D.shape[1]), dtype=np.float32)
    Sigma = np.zeros((num_classes, D.shape[1], D.shape[1]), dtype=np.float32)
    for k in range(num_classes):
        idx = np.where(y == k)[0]
        X = D[idx].astype(np.float32)
        mu[k] = X.mean(axis=0)
        C = np.cov(X.T, bias=False).astype(np.float32)
        # regularize
        C = C + reg * np.eye(C.shape[0], dtype=np.float32)
        Sigma[k] = C
    # Precompute inv(Sigma)
    Sigma_inv = np.linalg.inv(Sigma)
    return mu, Sigma, Sigma_inv

def cosine_scores(Z, P):
    """
    Z: (N,dim) embedding (not necessarily normalized)
    P: (K,dim) prototypes normalized
    return: cos sim (N,K)
    """
    ZN = l2_normalize(Z, axis=1)
    return ZN @ P.T

def mahalanobis(D, mu, Sigma_inv):
    """
    D: (N,d)
    mu: (d,)
    Sigma_inv: (d,d)
    return: (N,) distance
    """
    X = D - mu.reshape(1,-1)
    # d^2 = x^T S^-1 x
    # compute row-wise
    return np.einsum("nd,dd,nd->n", X, Sigma_inv, X).astype(np.float32)

def compute_double_scores(Z, Dfeat, P, mu_dom, Sigma_inv_dom):
    """
    Z: (N,emb), Dfeat: (N,D_DIM)
    return:
      S_id: (N,) max cosine to prototypes
      k_hat: (N,) argmax
      S_dom: (N,) Mahalanobis distance to predicted class domain stats
    """
    cos = cosine_scores(Z, P)                 # (N,K)
    k_hat = np.argmax(cos, axis=1).astype(np.int64)
    S_id = np.max(cos, axis=1).astype(np.float32)

    S_dom = np.zeros((Z.shape[0],), dtype=np.float32)
    for k in range(P.shape[0]):
        idx = np.where(k_hat == k)[0]
        if idx.size == 0:
            continue
        S_dom[idx] = mahalanobis(Dfeat[idx], mu_dom[k], Sigma_inv_dom[k])
    return S_id, k_hat, S_dom

In [14]:
def select_files(tx_list, rx_list, speed):
    return [r for r in file_records if (r["tx"] in tx_list and r["rx"] in rx_list and r["sp"] == speed)]

SOURCE_RXS = SOURCE_RX if isinstance(SOURCE_RX, (list, tuple, set)) else [SOURCE_RX]

files_known_drift = select_files(KNOWN_TX, EVAL_RX_LIST_DRIFT, EVAL_SPEED)
files_unknown_in = select_files(UNKNOWN_TX, SOURCE_RXS, EVAL_SPEED)
files_unknown_drift = select_files(UNKNOWN_TX, EVAL_RX_LIST_DRIFT, EVAL_SPEED)

print("Known-Drift files:", len(files_known_drift))
print("Unknown in-domain files:", len(files_unknown_in))
print("Unknown + drift files:", len(files_unknown_drift))

Known-Drift files: 12
Unknown in-domain files: 6
Unknown + drift files: 6


In [15]:
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_npz(path, **kwargs):
    np.savez_compressed(path, **kwargs)

def plot_hist_two(a, b, labels, title, out_png, bins=80):
    plt.figure(figsize=(6,4))
    plt.hist(a, bins=bins, alpha=0.6, density=True, label=labels[0])
    plt.hist(b, bins=bins, alpha=0.6, density=True, label=labels[1])
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def plot_roc(y_true, score, title, out_png):
    fpr, tpr, _ = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5,4))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    plt.plot([0,1], [0,1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(title); plt.grid(True); plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()
    return roc_auc

def plot_scatter_sid_sdom(S_id, S_dom, y, title, out_png, max_points=20000):
    n = len(S_id)
    if n > max_points:
        idx = np.random.choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(S_id[idx], S_dom[idx], s=3, c=y[idx], alpha=0.5)
    plt.xlabel("S_id (max cosine)")
    plt.ylabel("S_dom (Mahalanobis)")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

def load_eval_arrays(files, use_eq=True, max_per_file=0):
    """
    Return:
      Xeq_float: (N,256,2) float32  if use_eq
      Draw:      (N,D_DIM) float32  domain feats from raw always
      y_true:    (N,) int  for known/unknown discrimination:
                   known -> mapped class 0..K-1
                   unknown -> -1
    """
    X_list, D_list, y_list = [], [], []
    for rec in files:
        dmrs_raw = load_dmrs_raw_Nx256(rec["raw"])
        dmrs_eq  = load_dmrs_eq_Nx256(rec["eq"]) if use_eq else dmrs_raw
        assert dmrs_raw.shape == dmrs_eq.shape
        N = dmrs_eq.shape[0]
        if max_per_file and max_per_file > 0:
            N = min(N, max_per_file)
            dmrs_raw = dmrs_raw[:N]
            dmrs_eq  = dmrs_eq[:N]
        X_list.append(complex256_to_float2(dmrs_eq))
        D_list.append(np.stack([domain_feat_256(dmrs_raw[i]) for i in range(N)], axis=0))
        if rec["tx"] in KNOWN_TX:
            y_list.append(np.full((N,), KNOWN_TX.index(rec["tx"]), dtype=np.int64))
        else:
            y_list.append(np.full((N,), -1, dtype=np.int64))
    if len(X_list) == 0:
        return None, None, None
    X = np.concatenate(X_list, axis=0)
    Dd = np.concatenate(D_list, axis=0).astype(np.float32)
    yv = np.concatenate(y_list, axis=0).astype(np.int64)
    return X, Dd, yv

# -------- Cross-validation on blocks --------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_metrics = []

for fold, (blk_tr, blk_te) in enumerate(skf.split(unique_blocks, block_label), start=1):
    fold_dir = os.path.join(RUN_DIR, f"fold_{fold}")
    ensure_dir(fold_dir)

    # blocks -> sample indices
    tr_blocks = set(unique_blocks[blk_tr])
    te_blocks = set(unique_blocks[blk_te])
    idx_tr = np.where(np.isin(block_key, list(tr_blocks)))[0]
    idx_te = np.where(np.isin(block_key, list(te_blocks)))[0]

    # train/val split from train indices (simple)
    # 这里用 90/10 随机切分；更严格可再做 block-level split
    rng = np.random.RandomState(SEED + fold)
    perm = rng.permutation(idx_tr)
    n_val = int(0.1 * len(perm))
    idx_val = perm[:n_val]
    idx_train = perm[n_val:]

    X_train, y_train = X_eq[idx_train], y[idx_train]
    X_val,   y_val   = X_eq[idx_val],   y[idx_val]
    X_test,  y_test  = X_eq[idx_te],    y[idx_te]   # in-domain stable held-out

    train_loader = DataLoader(ArrayDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = DataLoader(ArrayDataset(X_val, y_val),     batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader  = DataLoader(ArrayDataset(X_test, y_test),   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = ResNet18_1D(num_classes=num_classes, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = -1.0
    best_state = None
    patience = 0
    history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

    for ep in range(1, NUM_EPOCHS+1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val + 1e-4:
            best_val = va_acc
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                break

    # save curves
    plot_curves(history, os.path.join(fold_dir, "train_curves.png"))
    with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    # load best
    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))

    # in-domain test acc + confmat
    model.eval()
    y_pred = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            y_pred.append(logits.argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=list(range(num_classes)))
    plot_confmat(cm, os.path.join(fold_dir, "confmat_in_domain.png"), title="In-domain (stable) Confusion")
    acc_in = float((y_pred == y_test).mean())

    # -------- prototypes & domain stats from TRAIN split (idx_train) --------
    Z_tr = compute_embeddings(model, X_eq[idx_train], batch_size=512)
    P = compute_prototypes(Z_tr, y[idx_train], num_classes)

    mu_dom, Sigma_dom, Sigma_inv_dom = compute_domain_stats(D[idx_train], y[idx_train], num_classes, reg=1e-3)

    save_npz(os.path.join(fold_dir, "prototypes_and_domain_stats.npz"),
             P=P, mu_dom=mu_dom, Sigma_inv_dom=Sigma_inv_dom)

    # -------- compute scores on scenarios --------
    # A) stable (in-domain test fold)
    Z_te = compute_embeddings(model, X_test, batch_size=512)
    D_te = D[idx_te]
    Sid_A, khat_A, Sdom_A = compute_double_scores(Z_te, D_te, P, mu_dom, Sigma_inv_dom)

    # B) known drift (cross RX, known tx)
    X_B, D_B, y_B = load_eval_arrays(files_known_drift, use_eq=True, max_per_file=MAX_SAMPLES_PER_FILE_EVAL)
    Z_B = compute_embeddings(model, X_B, batch_size=512)
    Sid_B, khat_B, Sdom_B = compute_double_scores(Z_B, D_B, P, mu_dom, Sigma_inv_dom)

    # C) unknown tx in-domain
    X_C, D_C, y_C = load_eval_arrays(files_unknown_in, use_eq=True, max_per_file=MAX_SAMPLES_PER_FILE_EVAL)
    Z_C = compute_embeddings(model, X_C, batch_size=512)
    Sid_C, khat_C, Sdom_C = compute_double_scores(Z_C, D_C, P, mu_dom, Sigma_inv_dom)

    # D) unknown tx + drift
    X_D, D_D, y_D = load_eval_arrays(files_unknown_drift, use_eq=True, max_per_file=MAX_SAMPLES_PER_FILE_EVAL)
    Z_D = compute_embeddings(model, X_D, batch_size=512)
    Sid_D, khat_D, Sdom_D = compute_double_scores(Z_D, D_D, P, mu_dom, Sigma_inv_dom)

    # -------- Save raw scores --------
    save_npz(os.path.join(fold_dir, "scores_stable_A.npz"),
             Sid=Sid_A, Sdom=Sdom_A, y=y_test, khat=khat_A)
    save_npz(os.path.join(fold_dir, "scores_known_drift_B.npz"),
             Sid=Sid_B, Sdom=Sdom_B, y=y_B, khat=khat_B)
    save_npz(os.path.join(fold_dir, "scores_unknown_in_C.npz"),
             Sid=Sid_C, Sdom=Sdom_C, y=y_C, khat=khat_C)
    save_npz(os.path.join(fold_dir, "scores_unknown_drift_D.npz"),
             Sid=Sid_D, Sdom=Sdom_D, y=y_D, khat=khat_D)

    # -------- Plots useful for paper --------
    # 1) S_id distribution: known stable vs unknown in-domain
    plot_hist_two(Sid_A, Sid_C, ["Known(stable)", "Unknown(in-domain)"],
                  "S_id distribution (Known vs Unknown)", os.path.join(fold_dir, "hist_Sid_known_vs_unknown.png"))
    # 2) S_dom distribution: stable vs known-drift (only meaningful when sample is known-ish; but先看全量)
    plot_hist_two(Sdom_A, Sdom_B, ["Stable(in-domain)", "Known-Drift(cross-RX)"],
                  "S_dom distribution (Stable vs Drift)", os.path.join(fold_dir, "hist_Sdom_stable_vs_drift.png"))

    # 3) ROC for unknown detection using -S_id (unknown=1)
    y_unknown = np.concatenate([np.zeros_like(Sid_A, dtype=np.int64), np.ones_like(Sid_C, dtype=np.int64)])
    score_unknown = np.concatenate([-Sid_A, -Sid_C])  # higher => more unknown
    auc_unknown = plot_roc(y_unknown, score_unknown, "Unknown detection ROC (score=-S_id)",
                           os.path.join(fold_dir, "roc_unknown_by_Sid.png"))

    # 4) ROC for drift detection using S_dom (drift=1) comparing A vs B
    y_drift = np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_B, dtype=np.int64)])
    score_drift = np.concatenate([Sdom_A, Sdom_B])  # higher => more drift
    auc_drift = plot_roc(y_drift, score_drift, "Drift detection ROC (score=S_dom)",
                         os.path.join(fold_dir, "roc_drift_by_Sdom.png"))

    # 5) 2D scatter S_id vs S_dom (four scenarios)
    # label: 0 stable, 1 drift, 2 unknown, 3 unknown+drift
    S_id_all = np.concatenate([Sid_A, Sid_B, Sid_C, Sid_D])
    S_dom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D])
    lab = np.concatenate([
        np.zeros_like(Sid_A, dtype=np.int64),
        np.ones_like(Sid_B, dtype=np.int64),
        np.full_like(Sid_C, 2, dtype=np.int64),
        np.full_like(Sid_D, 3, dtype=np.int64),
    ])
    plot_scatter_sid_sdom(S_id_all, S_dom_all, lab,
                          "S_id vs S_dom (0=stable,1=drift,2=unknown,3=unknown+drift)",
                          os.path.join(fold_dir, "scatter_Sid_Sdom.png"))

    # 6) simple combined diagnosis at fixed operating point (for analysis)
    # choose tau_id: 5% FRR on stable; tau_dom: 95% FPR on stable (false drift)
    tau_id = float(np.quantile(Sid_A, 0.05))
    tau_dom = float(np.quantile(Sdom_A, 0.95))
    # predict: 0 stable, 1 drift, 2 unknown
    def diagnose(Sid, Sdom):
        pred = np.zeros_like(Sid, dtype=np.int64)
        pred[Sid < tau_id] = 2
        pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
        return pred

    # ground truth for combined diagnosis:
    gt_A = np.zeros_like(Sid_A, dtype=np.int64)        # stable
    gt_B = np.ones_like(Sid_B, dtype=np.int64)         # drift
    gt_C = np.full_like(Sid_C, 2, dtype=np.int64)      # unknown
    gt_D = np.full_like(Sid_D, 2, dtype=np.int64)      # unknown (优先 unknown)

    pred_A = diagnose(Sid_A, Sdom_A)
    pred_B = diagnose(Sid_B, Sdom_B)
    pred_C = diagnose(Sid_C, Sdom_C)
    pred_D = diagnose(Sid_D, Sdom_D)

    gt_all = np.concatenate([gt_A, gt_B, gt_C, gt_D])
    pr_all = np.concatenate([pred_A, pred_B, pred_C, pred_D])
    cm3 = confusion_matrix(gt_all, pr_all, labels=[0,1,2])
    plot_confmat(cm3, os.path.join(fold_dir, "confmat_combined_diagnosis.png"),
                 title=f"Combined diagnosis @tau_id(p05),tau_dom(p95)")
    with open(os.path.join(fold_dir, "thresholds.json"), "w", encoding="utf-8") as f:
        json.dump({"tau_id_p05_stable": tau_id, "tau_dom_p95_stable": tau_dom}, f, indent=2)

    # record fold metrics
    fold_metrics.append({
        "fold": fold,
        "acc_in_domain": acc_in,
        "auc_unknown_by_Sid": float(auc_unknown),
        "auc_drift_by_Sdom": float(auc_drift),
        "tau_id": tau_id,
        "tau_dom": tau_dom
    })

    with open(os.path.join(fold_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(fold_metrics[-1], f, indent=2)

    print(f"[Fold {fold}] acc_in={acc_in:.3f}  AUC_unknown={auc_unknown:.3f}  AUC_drift={auc_drift:.3f}")

# Save summary
with open(os.path.join(RUN_DIR, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
    json.dump(fold_metrics, f, indent=2)

# Print average
acc_avg = np.mean([m["acc_in_domain"] for m in fold_metrics])
auc_u_avg = np.mean([m["auc_unknown_by_Sid"] for m in fold_metrics])
auc_d_avg = np.mean([m["auc_drift_by_Sdom"] for m in fold_metrics])
print("==== CV Summary ====")
print("acc_in_domain mean:", acc_avg)
print("auc_unknown mean:", auc_u_avg)
print("auc_drift mean:", auc_d_avg)

[Fold 1] acc_in=0.510  AUC_unknown=0.563  AUC_drift=0.602
[Fold 2] acc_in=0.535  AUC_unknown=0.517  AUC_drift=0.567
[Fold 3] acc_in=0.502  AUC_unknown=0.542  AUC_drift=0.606
[Fold 4] acc_in=0.537  AUC_unknown=0.550  AUC_drift=0.612
[Fold 5] acc_in=0.542  AUC_unknown=0.525  AUC_drift=0.623
==== CV Summary ====
acc_in_domain mean: 0.525300347470951
auc_unknown mean: 0.5395327100567627
auc_drift mean: 0.6019696305108609
